# DeepSORT Batch Evaluation Pipeline
This notebook clones your DeepSORT repository, processes a directory of multiple videos, and evaluates them collectively using TrackEval.

In [ ]:
# 1. Setup Environment
!git clone https://github.com/YolandCandy/DeepSORT.git
%cd DeepSORT
!pip install -r requirements.txt

In [ ]:
import os
import glob

# 2. Define Kaggle Dataset Paths
# UPDATE THESE to match the folders in your uploaded Kaggle dataset
VIDEOS_DIR = "/kaggle/input/your-dataset/videos"
# Assuming GT_DIR_INPUT points to the UA-DETRAC-test folder containing MVI_39031/gt/gt.txt etc.
GT_DIR_INPUT = "/kaggle/input/your-dataset/TrackEval/data/gt/mot_challenge/UA-DETRAC-test"

BENCHMARK = "UA-DETRAC"
SPLIT = "test"
TRACKER = "deepsort"

video_files = glob.glob(os.path.join(VIDEOS_DIR, "*.mp4"))
print(f"Found {len(video_files)} videos to process.")

In [ ]:
# 3. Batch Process Videos and Setup TrackEval Format
import cv2
import numpy as np
import os

seqmap_dir = f"TrackEval/data/gt/mot_challenge/seqmaps"
os.makedirs(seqmap_dir, exist_ok=True)
seqmap_path = f"{seqmap_dir}/{BENCHMARK}-{SPLIT}.txt"

with open(seqmap_path, "w") as seqmap_file:
    seqmap_file.write("name\n")
    
    for video_path in video_files:
        seq_name = os.path.splitext(os.path.basename(video_path))[0]
        seqmap_file.write(f"{seq_name}\n")
        print(f"\n--- Processing Sequence: {seq_name} ---")
        
        # Define output paths for deepsort
        tracking_out = f"/kaggle/working/DeepSORT/output_trackeval_{seq_name}.txt"
        output_video = f"/kaggle/working/DeepSORT/output_video_{seq_name}.mp4"
        
        # ---------------------------------------------------------
        # Pre-process video to paint ignore regions black
        # ---------------------------------------------------------
        ignore_txt = f"TrackEval/data/ignore_region/TrackEval/data/{seq_name}/ignored_regions.txt"
        temp_video_path = video_path
        
        if os.path.exists(ignore_txt):
            print(f"Applying ignore regions from {ignore_txt}")
            with open(ignore_txt, 'r') as f:
                ignore_boxes = [list(map(float, line.strip().split(','))) for line in f if line.strip()]
            
            temp_video_path = f"/kaggle/working/temp_masked_{seq_name}.mp4"
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(temp_video_path, fourcc, fps, (w, h))
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                for box in ignore_boxes:
                    l, t, bw, bh = map(int, box)
                    # Paint black to avoid detection in ignored region
                    cv2.rectangle(frame, (l, t), (l+bw, t+bh), (0, 0, 0), -1)
                out.write(frame)
            cap.release()
            out.release()
        
        # RUN DeepSORT Tracker
        save_vid_arg = "--save-video" if seq_name == "MVI_39031" else ""
        !python deepsort.py --video {temp_video_path} --output {output_video} --trackeval-output {tracking_out} {save_vid_arg}
        
        # Clean up temporary video
        if temp_video_path != video_path and os.path.exists(temp_video_path):
            os.remove(temp_video_path)
            
        # Prepare GT folder structure for this sequence
        dest_gt = f"TrackEval/data/gt/mot_challenge/{BENCHMARK}-{SPLIT}/"
        os.makedirs(dest_gt, exist_ok=True)
        
        # Copy the entire sequence folder from the Kaggle dataset to TrackEval
        seq_gt_input = os.path.join(GT_DIR_INPUT, seq_name)
        if os.path.exists(seq_gt_input):
            !cp -r {seq_gt_input} {dest_gt}
        else:
            print(f"[WARNING] Ground truth folder not found: {seq_gt_input}")
            
        # Prepare Tracker folder structure and copy tracking output
        tracker_dir = f"TrackEval/data/trackers/mot_challenge/{BENCHMARK}-{SPLIT}/{TRACKER}/data"
        os.makedirs(tracker_dir, exist_ok=True)
        
        if os.path.exists(tracking_out):
            !cp {tracking_out} {tracker_dir}/{seq_name}.txt
            
print("\nAll videos processed. TrackEval environment is fully formatted.")



In [ ]:
# 4. Run TrackEval
# This will automatically evaluate all sequences listed in the seqmap file together.
!python TrackEval/scripts/run_mot_challenge.py --BENCHMARK {BENCHMARK} --TRACKERS_TO_EVAL {TRACKER} --SPLIT_TO_EVAL {SPLIT} --METRICS HOTA CLEAR Identity --USE_PARALLEL False --NUM_PARALLEL_CORES 1